# AttackV2 custom epsilon accuracy curves

Targeted comparison view: choose one primary model/group as source or target, choose the comparison groups on the opposite side, and draw robust-accuracy curves over FGSM epsilon from stored attack aggregate tables.

## 1. Imports, utilities, and config

In [ ]:
from pathlib import Path
import sys

try:
    from analytics.paths import attack_experiment_dir, find_project_root
except ModuleNotFoundError:
    repo_root = Path.cwd().resolve()
    while repo_root != repo_root.parent and not (repo_root / "src" / "analytics").is_dir():
        repo_root = repo_root.parent
    sys.path.insert(0, str(repo_root / "src"))
    from analytics.paths import attack_experiment_dir, find_project_root

from analytics.attacks.attacksv2_transfer_tools import (
    AttacksV2Config,
    export_figure,
    export_table,
    v2_output_dir,
)
from analytics.attacks.attacksv2_accuracy_curves import (
    ROBUST_METRIC,
    available_curve_groups,
    build_custom_accuracy_curve_table,
    load_attack_curve_rows,
    plot_accuracy_curves,
    prepare_curve_rows,
)

ROOT = find_project_root()
ATTACK_DIR = attack_experiment_dir(ROOT)
OUT_DIR = v2_output_dir("curves_custom")

def safe_slug(value: str) -> str:
    return (
        str(value)
        .replace("/", "_")
        .replace(" ", "_")
        .replace(",", "")
        .replace("=", "-")
        .replace("[", "")
        .replace("]", "")
    )


In [ ]:
# Scenario: CNN as attack source, compare selected target variants.
# Use inter_variant so a CNN source group can still be compared against a different CNN target variant.
CONFIG = AttacksV2Config(
    source_grain="architecture",
    target_grain="variant",
    comparison_mode="inter_variant",
    metric=ROBUST_METRIC,
    variant_aggregate_mode="mean",
    architecture_aggregate_mode="mean",
    # split_quantum_ansatz_as_architecture=True,
    # blacklist_contains=("HQuanv",),
)

# Flip to PRIMARY_ROLE = "target" when the selected model/group should be the victim.
PRIMARY_ROLE = "source"
PRIMARY_GROUP = "Classical/CNN"
COMPARISON_GROUPS = (
    "Classical/MLP/hidden_sizes=[4,2]",              # weakest MLP target
    "Classical/CNN/conv_channels=[8], fc_sizes=[10]", # weakest CNN target
    "Quantum/VQM/ansatz=strong, qdepth=200",          # strongest VQM target
)
INCLUDE_SELF_ATTACKS = False

CONFIG

## 2. Crawl and find attack tables

In [ ]:
TABLE_FILES = sorted((ATTACK_DIR / "tables").glob("*.parquet")) or sorted((ATTACK_DIR / "tables").glob("*.csv"))
print(f"Attack dir: {ATTACK_DIR}")
print(f"Tables found: {len(TABLE_FILES)}")
TABLE_FILES[:5]

## 3. Load rows and inspect available groups

In [ ]:
raw_df = load_attack_curve_rows(ATTACK_DIR)
print(f"Raw aggregate rows: {raw_df.shape[0]:,} x {raw_df.shape[1]:,}")

groups = available_curve_groups(raw_df, CONFIG, include_self_attacks=INCLUDE_SELF_ATTACKS)
print("Available source groups")
display(groups["source"])
print("Available target groups")
display(groups["target"])

prepare_curve_rows(raw_df, CONFIG, include_self_attacks=INCLUDE_SELF_ATTACKS).head()

## 4. Generate targeted comparison figure/table

In [ ]:
custom_curve = build_custom_accuracy_curve_table(
    raw_df,
    CONFIG,
    primary_role=PRIMARY_ROLE,
    primary_group=PRIMARY_GROUP,
    comparison_groups=COMPARISON_GROUPS,
    include_self_attacks=INCLUDE_SELF_ATTACKS,
)
custom_curve

In [ ]:
name = f"custom_curves__{PRIMARY_ROLE}__{safe_slug(PRIMARY_GROUP)}__{CONFIG.slug()}"
table_paths = export_table(custom_curve, OUT_DIR, name)

comparison_side = "targets" if PRIMARY_ROLE == "source" else "sources"
fig, ax = plot_accuracy_curves(
    custom_curve,
    CONFIG,
    mode=PRIMARY_ROLE,
    selected_group=PRIMARY_GROUP,
    title=f"{PRIMARY_GROUP} as {PRIMARY_ROLE}: selected {comparison_side}",
)
figure_paths = export_figure(fig, OUT_DIR, name)

{"table": table_paths, "figure": figure_paths}